# DELTA DIY MRI Workshop — Environment Setup

This notebook creates **two clean Python 3.11 environments** for the DELTA DIY MRI workshop.

We keep these separate because the acquisition notebooks and passive-shimming notebook currently need different NumPy families:

| Environment | Kernel name | Purpose | NumPy requirement |
|---|---|---|---|
| `.diy-mri-workshop` | `DIY MRI Workshop Python 3.11` | RF SE, pulse-width calibration, 1D SE, 2D SE, PyPulseq, MRZeroCore | `numpy<2` |
| `.diy-mri-shimming` | `DIY MRI Passive Shimming Python 3.11` | passive shimming, pymoo optimization, Magpylib, STL tray visualization/export | `numpy>=2.0` |

This avoids the package conflict:

```text
passive-shimming==0.0.1 requires numpy>=2.0
PyPulseq / MRZero workshop environment is kept on numpy<2
```

**Recommended macOS choice:** use the Python 3.11 installer from `python.org`, not Homebrew Python, for the workshop environment. During beta testing this avoided Homebrew-specific `pyexpat/libexpat`, `pip`, `matplotlib`, PyPulseq, and MRZeroCore issues on macOS 26.x.

**How to use this notebook:** most setup cells print Terminal / Command Prompt commands. Copy and run those commands outside the notebook. Then return to this notebook, select the requested kernel, and run the verification cells.

In [11]:
# Detect operating system and CPU architecture

import platform
import sys
from pathlib import Path

print("System detection")
print("----------------")

os_name = platform.system()
machine = platform.machine().lower()
repo_dir = Path.cwd()

if os_name == "Darwin":
    if machine in ["arm64", "aarch64"]:
        setup_category = "macOS Apple Silicon"
    elif machine in ["x86_64", "amd64"]:
        setup_category = "macOS Intel"
    else:
        setup_category = "macOS unknown CPU"
elif os_name == "Windows":
    setup_category = "Windows"
elif os_name == "Linux":
    setup_category = "Linux"
else:
    setup_category = "Unknown"

print("Current notebook Python:")
print("  Executable:", sys.executable)
print("  Version   :", sys.version.split()[0])

print("\nSystem:")
print("  OS        :", os_name)
print("  Platform  :", platform.platform())
print("  Machine   :", platform.machine())
print("  Processor :", platform.processor())
print("  Category  :", setup_category)
print("\nRepository folder:")
print(" ", repo_dir)

System detection
----------------
Current notebook Python:
  Executable: /Users/sairamgeethanath/Documents/Contributions/Tools/Projects/adelpha/.diy-mri-workshop/bin/python
  Version   : 3.11.9

System:
  OS        : Darwin
  Platform  : macOS-26.6.2-arm64-arm-64bit
  Machine   : arm64
  Processor : arm
  Category  : macOS Apple Silicon

Repository folder:
  /Users/sairamgeethanath/Documents/Contributions/Tools/Projects/adelpha/console/notebooks


## Check Python 3.11

Run this cell to print the recommended Python 3.11 check and install instructions for your system.

For macOS, the workshop-recommended path is the `python.org` installer. If a Homebrew Python passes `python3.11 --version` but fails `xml.parsers.expat`, do not use it for the workshop.

In [13]:
# Check for Python 3.11 and print install instructions

import platform
import subprocess

print("Python 3.11 check")
print("-----------------")

os_name = platform.system()

if os_name == "Darwin":
    python311_candidate = "/usr/local/bin/python3.11"
    check_cmd = [python311_candidate, "--version"]
    install_instructions = """
macOS install instructions
--------------------------

Recommended for this workshop:
1. Open: https://www.python.org/downloads/
2. Download a Python 3.11.x macOS universal2 installer.
3. Run the .pkg installer.
4. Open a new Terminal window.
5. Verify:

   /usr/local/bin/python3.11 --version
   /usr/local/bin/python3.11 -c "import xml.parsers.expat as expat; print('pyexpat OK')"
   /usr/local/bin/python3.11 -c "import platform; print(platform.mac_ver())"

Avoid Homebrew Python for the workshop if pyexpat or pip fails.
"""
elif os_name == "Windows":
    python311_candidate = "py -3.11"
    check_cmd = ["py", "-3.11", "--version"]
    install_instructions = """
Windows install instructions
----------------------------

Recommended for this workshop:
1. Open: https://www.python.org/downloads/windows/
2. Download a Python 3.11.x Windows installer.
3. During installation, check: Add python.exe to PATH.
4. Open a new Command Prompt.
5. Verify:

   py -3.11 --version

If the py launcher is unavailable, try:

   python --version
"""
elif os_name == "Linux":
    python311_candidate = "python3.11"
    check_cmd = ["python3.11", "--version"]
    install_instructions = """
Linux install instructions
--------------------------

Install Python 3.11 using your distribution package manager.

Ubuntu / Debian example:
   sudo apt update
   sudo apt install python3.11 python3.11-venv python3.11-dev

Then verify:
   python3.11 --version
"""
else:
    python311_candidate = "python3.11"
    check_cmd = ["python3.11", "--version"]
    install_instructions = "Unknown OS. Please install Python 3.11 and make sure python3.11 is available."

print("Detected OS             :", os_name)
print("Python 3.11 candidate   :", python311_candidate)
print()

try:
    result = subprocess.run(check_cmd, capture_output=True, text=True)
    print("Check command:", " ".join(check_cmd))
    print("Return code  :", result.returncode)
    print("stdout       :", result.stdout.strip())
    print("stderr       :", result.stderr.strip())
    if result.returncode == 0:
        print("\nPASS: Python 3.11 command is available.")
    else:
        print("\nWARNING: Python 3.11 command was not found or did not run.")
except Exception as exc:
    print("WARNING: Python 3.11 check could not run:", repr(exc))

print("\n" + install_instructions)

Python 3.11 check
-----------------
Detected OS             : Darwin
Python 3.11 candidate   : /usr/local/bin/python3.11

Check command: /usr/local/bin/python3.11 --version
Return code  : 0
stdout       : Python 3.11.9
stderr       : 

PASS: Python 3.11 command is available.


macOS install instructions
--------------------------

Recommended for this workshop:
1. Open: https://www.python.org/downloads/
2. Download a Python 3.11.x macOS universal2 installer.
3. Run the .pkg installer.
4. Open a new Terminal window.
5. Verify:

   /usr/local/bin/python3.11 --version
   /usr/local/bin/python3.11 -c "import xml.parsers.expat as expat; print('pyexpat OK')"
   /usr/local/bin/python3.11 -c "import platform; print(platform.mac_ver())"

Avoid Homebrew Python for the workshop if pyexpat or pip fails.



## Create `.diy-mri-workshop` for acquisition / console notebooks

This environment is for RF spin echo, pulse-width calibration, 1D SE, 2D SE, PyPulseq `.seq` export, and MRZeroCore simulation.

It intentionally uses `numpy<2` and **does not** install `passive-shimming`.

In [14]:
# Print commands to create the acquisition / console virtual environment

import platform
from pathlib import Path

print("Create acquisition / console environment")
print("----------------------------------------")

os_name = platform.system()
repo_dir = Path.cwd()
venv_name = ".diy-mri-workshop"

try:
    python311 = python311_candidate
except NameError:
    if os_name == "Darwin":
        python311 = "/usr/local/bin/python3.11"
    elif os_name == "Windows":
        python311 = "py -3.11"
    else:
        python311 = "python3.11"

print("Detected OS:", os_name)
print("Repository folder:", repo_dir)
print("Environment folder:", repo_dir / venv_name)
print("Python 3.11 command:", python311)

print("\nRun these commands in Terminal / Command Prompt.")
print("-----------------------------------------------")

if os_name == "Darwin":
    print(f"""
cd "{repo_dir}"

deactivate 2>/dev/null || true
unset PIP_USE_DEPRECATED

rm -rf {venv_name}

{python311} -m venv {venv_name}

{venv_name}/bin/python --version
{venv_name}/bin/python -m pip --version
{venv_name}/bin/python -c "import xml.parsers.expat as expat; print('pyexpat OK')"
{venv_name}/bin/python -c "import platform; print('mac_ver:', platform.mac_ver())"
""")
elif os_name == "Windows":
    print(f"""
cd "{repo_dir}"

REM Clean rebuild. Ignore errors if the folder does not exist.
rmdir /s /q {venv_name}

{python311} -m venv {venv_name}

{venv_name}\\Scripts\\python --version
{venv_name}\\Scripts\\python -m pip --version
""")
else:
    print(f"""
cd "{repo_dir}"

deactivate 2>/dev/null || true
rm -rf {venv_name}

{python311} -m venv {venv_name}

{venv_name}/bin/python --version
{venv_name}/bin/python -m pip --version
""")

Create acquisition / console environment
----------------------------------------
Detected OS: Darwin
Repository folder: /Users/sairamgeethanath/Documents/Contributions/Tools/Projects/adelpha/console/notebooks
Environment folder: /Users/sairamgeethanath/Documents/Contributions/Tools/Projects/adelpha/console/notebooks/.diy-mri-workshop
Python 3.11 command: /usr/local/bin/python3.11

Run these commands in Terminal / Command Prompt.
-----------------------------------------------

cd "/Users/sairamgeethanath/Documents/Contributions/Tools/Projects/adelpha/console/notebooks"

deactivate 2>/dev/null || true
unset PIP_USE_DEPRECATED

rm -rf .diy-mri-workshop

/usr/local/bin/python3.11 -m venv .diy-mri-workshop

.diy-mri-workshop/bin/python --version
.diy-mri-workshop/bin/python -m pip --version
.diy-mri-workshop/bin/python -c "import xml.parsers.expat as expat; print('pyexpat OK')"
.diy-mri-workshop/bin/python -c "import platform; print('mac_ver:', platform.mac_ver())"



In [15]:
# Print commands to install acquisition / console packages

import platform
from pathlib import Path

print("Install acquisition / console packages")
print("--------------------------------------")

os_name = platform.system()
repo_dir = Path.cwd()
venv_name = ".diy-mri-workshop"
kernel_name = "diy-mri-workshop"
kernel_display_name = "DIY MRI Workshop Python 3.11"

acquisition_packages = [
    "numpy<2",
    "matplotlib",
    "scipy",
    "torch",
    "pypulseq==1.4.2.post1",
    "MRzeroCore",
    "brainweb-dl",
    "jupyter",
    "ipykernel",
]

print("Packages:")
for package in acquisition_packages:
    print(" ", package)

print("\nRun these commands in Terminal / Command Prompt.")
print("-----------------------------------------------")

if os_name == "Windows":
    py = f"{venv_name}\\Scripts\\python"
    package_block = " ^\n  ".join(f'"{p}"' for p in acquisition_packages)
    print(f"""
cd "{repo_dir}"

{py} -m pip install --upgrade pip setuptools wheel

{py} -m pip install ^
  {package_block}

{py} -m ipykernel install --user ^
  --name {kernel_name} ^
  --display-name "{kernel_display_name}"

{py} -c "import sys; print('Python executable:', sys.executable); print('Python version:', sys.version)"
{py} -c "import numpy as np; print('NumPy OK:', np.__version__)"
{py} -c "import pypulseq as pp; print('PyPulseq OK:', pp.__file__)"
{py} -c "import MRzeroCore as mr0; print('MRZeroCore OK:', mr0.__file__)"
""")
else:
    py = f"{venv_name}/bin/python"
    package_block = " \\\n  ".join(f'"{p}"' for p in acquisition_packages)
    print(f"""
cd "{repo_dir}"

{py} -m pip install --upgrade pip setuptools wheel

{py} -m pip install \\
  {package_block}

{py} -m ipykernel install --user \\
  --name {kernel_name} \\
  --display-name "{kernel_display_name}"

{py} -c "import sys; print('Python executable:', sys.executable); print('Python version:', sys.version)"
{py} -c "import numpy as np; print('NumPy OK:', np.__version__)"
{py} -c "import pypulseq as pp; print('PyPulseq OK:', pp.__file__)"
{py} -c "import MRzeroCore as mr0; print('MRZeroCore OK:', mr0.__file__)"
""")

print("\nAfter this succeeds, select this notebook kernel:")
print(" ", kernel_display_name)

Install acquisition / console packages
--------------------------------------
Packages:
  numpy<2
  matplotlib
  scipy
  torch
  pypulseq==1.4.2.post1
  MRzeroCore
  brainweb-dl
  jupyter
  ipykernel

Run these commands in Terminal / Command Prompt.
-----------------------------------------------

cd "/Users/sairamgeethanath/Documents/Contributions/Tools/Projects/adelpha/console/notebooks"

.diy-mri-workshop/bin/python -m pip install --upgrade pip setuptools wheel

.diy-mri-workshop/bin/python -m pip install \
  "numpy<2" \
  "matplotlib" \
  "scipy" \
  "torch" \
  "pypulseq==1.4.2.post1" \
  "MRzeroCore" \
  "brainweb-dl" \
  "jupyter" \
  "ipykernel"

.diy-mri-workshop/bin/python -m ipykernel install --user \
  --name diy-mri-workshop \
  --display-name "DIY MRI Workshop Python 3.11"

.diy-mri-workshop/bin/python -c "import sys; print('Python executable:', sys.executable); print('Python version:', sys.version)"
.diy-mri-workshop/bin/python -c "import numpy as np; print('NumPy OK:', 

## Verify `.diy-mri-workshop`

Select the kernel:

```text
DIY MRI Workshop Python 3.11
```

Then restart the notebook kernel and run the next verification cells.

In [ ]:
# Verify acquisition / console notebook kernel

import sys
import platform

print("Acquisition / console kernel verification")
print("-----------------------------------------")

print("Python executable:", sys.executable)
print("Python version   :", sys.version)
print("Platform         :", platform.platform())
print("mac_ver          :", platform.mac_ver())

expected_env_name = ".diy-mri-workshop"

if expected_env_name in sys.executable:
    print("\nPASS: This notebook is using the acquisition / console environment.")
else:
    print("\nWARNING: This notebook is not using the expected environment.")
    print("Select the kernel:")
    print("  DIY MRI Workshop Python 3.11")
    print("Then restart the notebook kernel and rerun this cell.")

In [16]:
# Acquisition / console package smoke test

print("Acquisition / console package smoke test")
print("----------------------------------------")

import sys
from importlib.metadata import version, PackageNotFoundError

import numpy as np
import matplotlib
import scipy
import torch
import pypulseq as pp
import MRzeroCore as mr0

print("Python executable:", sys.executable)

print("\nCore packages")
print("-------------")
print("NumPy      :", np.__version__)
print("Matplotlib :", matplotlib.__version__)
print("SciPy      :", scipy.__version__)
print("Torch      :", torch.__version__)

print("\nMRI packages")
print("------------")
print("PyPulseq   :", pp.__file__)
print("MRZeroCore :", mr0.__file__)

major_numpy = int(np.__version__.split('.')[0])
if major_numpy >= 2:
    print("\nWARNING: NumPy is 2.x. This environment is expected to use numpy<2.")
else:
    print("\nPASS: Acquisition environment has numpy<2.")

try:
    version("passive-shimming")
    print("\nWARNING: passive-shimming is installed here, but it should live in .diy-mri-shimming.")
except PackageNotFoundError:
    print("PASS: passive-shimming is not installed in the acquisition environment.")

Acquisition / console package smoke test
----------------------------------------
Python executable: /Users/sairamgeethanath/Documents/Contributions/Tools/Projects/adelpha/.diy-mri-workshop/bin/python

Core packages
-------------
NumPy      : 1.26.4
Matplotlib : 3.11.1
SciPy      : 1.17.1
Torch      : 2.14.0

MRI packages
------------
PyPulseq   : /Users/sairamgeethanath/Documents/Contributions/Tools/Projects/adelpha/.diy-mri-workshop/lib/python3.11/site-packages/pypulseq/__init__.py
MRZeroCore : /Users/sairamgeethanath/Documents/Contributions/Tools/Projects/adelpha/.diy-mri-workshop/lib/python3.11/site-packages/MRzeroCore/__init__.py

PASS: Acquisition environment has numpy<2.
PASS: passive-shimming is not installed in the acquisition environment.


In [17]:
# PyPulseq .seq export smoke test
# This checks that PyPulseq can create and export a simple .seq file.
# This is not a scanner-ready MRI sequence.

print("PyPulseq sequence export smoke test")
print("-----------------------------------")

import numpy as np
import pypulseq as pp
from pathlib import Path

BLOCK_RASTER_TIME = 10e-6

def round_to_raster(t, raster=BLOCK_RASTER_TIME):
    """Round time in seconds to nearest raster point."""
    t_rounded = round(t / raster) * raster
    if t_rounded < 0:
        raise ValueError(f"Negative delay after rounding: {t_rounded*1e6:.1f} µs")
    return t_rounded

system = pp.Opts(
    max_grad=10,
    grad_unit="mT/m",
    max_slew=50,
    slew_unit="T/m/s",
    rf_ringdown_time=20e-6,
    rf_dead_time=100e-6,
    adc_dead_time=10e-6,
)

seq = pp.Sequence(system)

rf_duration = round_to_raster(200e-6)
delay_duration = round_to_raster(10e-3)

rf = pp.make_block_pulse(
    flip_angle=np.pi / 2,
    duration=rf_duration,
    delay=system.rf_dead_time,
    system=system,
)

seq.add_block(rf)
seq.add_block(pp.make_delay(delay_duration))

ok, error_report = seq.check_timing()

if ok:
    print("PASS: PyPulseq timing check passed.")
else:
    print("WARNING: PyPulseq timing check failed.")
    print(error_report)

out_file = Path("pypulseq_smoke_test.seq")
seq.write(str(out_file))

if out_file.exists():
    print("PASS: Wrote", out_file.resolve())
    print("File size:", out_file.stat().st_size, "bytes")
else:
    print("WARNING: .seq file was not found after write.")

PyPulseq sequence export smoke test
-----------------------------------
PASS: PyPulseq timing check passed.
PASS: Wrote /Users/sairamgeethanath/Documents/Contributions/Tools/Projects/adelpha/console/notebooks/pypulseq_smoke_test.seq
File size: 1048 bytes


In [18]:
# MRZeroCore smoke test and minimal API check

print("MRZeroCore smoke test")
print("---------------------")

import inspect
from importlib.metadata import version
import MRzeroCore as mr0

print("MRZeroCore imported successfully.")
print("MRZeroCore version:", version("MRzeroCore"))
print("MRZeroCore path   :", mr0.__file__)

expected_names = [
    "Sequence",
    "VoxelGridPhantom",
    "CustomVoxelPhantom",
    "compute_graph",
    "execute_graph",
    "reco_adjoint",
]

print("\nExpected object check")
print("---------------------")

all_found = True

for name in expected_names:
    if hasattr(mr0, name):
        obj = getattr(mr0, name)
        print(f"PASS: mr0.{name} found")
        try:
            print("      Signature:", inspect.signature(obj))
        except Exception:
            pass
    else:
        all_found = False
        print(f"WARNING: mr0.{name} not found")

if all_found:
    print("\nPASS: MRZeroCore has the expected objects for the workshop notebooks.")
else:
    print("\nWARNING: Some expected MRZeroCore objects were not found.")

MRZeroCore smoke test
---------------------
MRZeroCore imported successfully.
MRZeroCore version: 1.0.8
MRZeroCore path   : /Users/sairamgeethanath/Documents/Contributions/Tools/Projects/adelpha/.diy-mri-workshop/lib/python3.11/site-packages/MRzeroCore/__init__.py

Expected object check
---------------------
PASS: mr0.Sequence found
      Signature: (repetitions: 'Iterable[Repetition]' = [], normalized_grads: 'bool' = True)
PASS: mr0.VoxelGridPhantom found
      Signature: (PD: 'torch.Tensor', T1: 'torch.Tensor', T2: 'torch.Tensor', T2dash: 'torch.Tensor', D: 'torch.Tensor', B0: 'torch.Tensor', B1: 'torch.Tensor', coil_sens: 'torch.Tensor', affine: 'torch.Tensor', phantom_motion=None, voxel_motion=None, tissue_masks: 'Optional[Dict[str, torch.Tensor]]' = None) -> 'None'
PASS: mr0.CustomVoxelPhantom found
      Signature: (pos: 'list[list[float]] | torch.Tensor', PD: 'float | list[float] | torch.Tensor' = 1.0, T1: 'float | list[float] | torch.Tensor' = 1.5, T2: 'float | list[float] | to

## Create `.diy-mri-shimming` for passive shimming

This environment is for `passive-shimming==0.0.1`, `pymoo`, `magpylib`, STL inspection/visualization, and the passive-shimming notebook.

It intentionally uses `numpy>=2.0` and is kept separate from `.diy-mri-workshop`.

In [19]:
# Print commands to create the passive-shimming virtual environment

import platform
from pathlib import Path

print("Create passive-shimming environment")
print("-----------------------------------")

os_name = platform.system()
repo_dir = Path.cwd()
venv_name = ".diy-mri-shimming"

try:
    python311 = python311_candidate
except NameError:
    if os_name == "Darwin":
        python311 = "/usr/local/bin/python3.11"
    elif os_name == "Windows":
        python311 = "py -3.11"
    else:
        python311 = "python3.11"

print("Detected OS:", os_name)
print("Repository folder:", repo_dir)
print("Environment folder:", repo_dir / venv_name)
print("Python 3.11 command:", python311)

print("\nRun these commands in Terminal / Command Prompt.")
print("-----------------------------------------------")

if os_name == "Darwin":
    print(f"""
cd "{repo_dir}"

deactivate 2>/dev/null || true
unset PIP_USE_DEPRECATED

rm -rf {venv_name}

{python311} -m venv {venv_name}

{venv_name}/bin/python --version
{venv_name}/bin/python -m pip --version
{venv_name}/bin/python -c "import xml.parsers.expat as expat; print('pyexpat OK')"
{venv_name}/bin/python -c "import platform; print('mac_ver:', platform.mac_ver())"
""")
elif os_name == "Windows":
    print(f"""
cd "{repo_dir}"

REM Clean rebuild. Ignore errors if the folder does not exist.
rmdir /s /q {venv_name}

{python311} -m venv {venv_name}

{venv_name}\\Scripts\\python --version
{venv_name}\\Scripts\\python -m pip --version
""")
else:
    print(f"""
cd "{repo_dir}"

deactivate 2>/dev/null || true
rm -rf {venv_name}

{python311} -m venv {venv_name}

{venv_name}/bin/python --version
{venv_name}/bin/python -m pip --version
""")

Create passive-shimming environment
-----------------------------------
Detected OS: Darwin
Repository folder: /Users/sairamgeethanath/Documents/Contributions/Tools/Projects/adelpha/console/notebooks
Environment folder: /Users/sairamgeethanath/Documents/Contributions/Tools/Projects/adelpha/console/notebooks/.diy-mri-shimming
Python 3.11 command: /usr/local/bin/python3.11

Run these commands in Terminal / Command Prompt.
-----------------------------------------------

cd "/Users/sairamgeethanath/Documents/Contributions/Tools/Projects/adelpha/console/notebooks"

deactivate 2>/dev/null || true
unset PIP_USE_DEPRECATED

rm -rf .diy-mri-shimming

/usr/local/bin/python3.11 -m venv .diy-mri-shimming

.diy-mri-shimming/bin/python --version
.diy-mri-shimming/bin/python -m pip --version
.diy-mri-shimming/bin/python -c "import xml.parsers.expat as expat; print('pyexpat OK')"
.diy-mri-shimming/bin/python -c "import platform; print('mac_ver:', platform.mac_ver())"



In [20]:
# Print commands to install passive-shimming packages

import platform
from pathlib import Path

print("Install passive-shimming packages")
print("---------------------------------")

os_name = platform.system()
repo_dir = Path.cwd()
venv_name = ".diy-mri-shimming"
kernel_name = "diy-mri-shimming"
kernel_display_name = "DIY MRI Passive Shimming Python 3.11"

shimming_packages = [
    "numpy>=2.0",
    "passive-shimming==0.0.1",
    "pymoo",
    "magpylib",
    "trimesh",
    "numpy-stl",
    "plotly",
    "matplotlib",
    "scipy",
    "jupyter",
    "ipykernel",
]

print("Packages:")
for package in shimming_packages:
    print(" ", package)

print("\nRun these commands in Terminal / Command Prompt.")
print("-----------------------------------------------")

if os_name == "Windows":
    py = f"{venv_name}\\Scripts\\python"
    package_block = " ^\n  ".join(f'"{p}"' for p in shimming_packages)
    print(f"""
cd "{repo_dir}"

{py} -m pip install --upgrade pip setuptools wheel

{py} -m pip install ^
  {package_block}

{py} -m ipykernel install --user ^
  --name {kernel_name} ^
  --display-name "{kernel_display_name}"

{py} -c "import sys; print('Python executable:', sys.executable); print('Python version:', sys.version)"
{py} -c "import numpy as np; print('NumPy OK:', np.__version__)"
{py} -c "import pymoo; print('pymoo OK:', pymoo.__version__)"
{py} -c "import magpylib as magpy; print('magpylib OK:', magpy.__version__)"
{py} -c "import trimesh; print('trimesh OK:', trimesh.__version__)"
{py} -c "import stl; print('numpy-stl OK')"
{py} -c "import plotly; print('plotly OK:', plotly.__version__)"
{py} -c "import shimming_io; print('shimming_io OK')"
{py} -c "import shimming_geometry; print('shimming_geometry OK')"
{py} -c "import shimming_basis; print('shimming_basis OK')"
{py} -c "import shimming_optimization; print('shimming_optimization OK')"
{py} -c "import shimming_reporting; print('shimming_reporting OK')"
{py} -c "import shimming_export; print('shimming_export OK')"
""")
else:
    py = f"{venv_name}/bin/python"
    package_block = " \\\n  ".join(f'"{p}"' for p in shimming_packages)
    print(f"""
cd "{repo_dir}"

{py} -m pip install --upgrade pip setuptools wheel

{py} -m pip install \\
  {package_block}

{py} -m ipykernel install --user \\
  --name {kernel_name} \\
  --display-name "{kernel_display_name}"

{py} -c "import sys; print('Python executable:', sys.executable); print('Python version:', sys.version)"
{py} -c "import numpy as np; print('NumPy OK:', np.__version__)"
{py} -c "import pymoo; print('pymoo OK:', pymoo.__version__)"
{py} -c "import magpylib as magpy; print('magpylib OK:', magpy.__version__)"
{py} -c "import trimesh; print('trimesh OK:', trimesh.__version__)"
{py} -c "import stl; print('numpy-stl OK')"
{py} -c "import plotly; print('plotly OK:', plotly.__version__)"
{py} -c "import shimming_io; print('shimming_io OK')"
{py} -c "import shimming_geometry; print('shimming_geometry OK')"
{py} -c "import shimming_basis; print('shimming_basis OK')"
{py} -c "import shimming_optimization; print('shimming_optimization OK')"
{py} -c "import shimming_reporting; print('shimming_reporting OK')"
{py} -c "import shimming_export; print('shimming_export OK')"
""")

print("\nAfter this succeeds, select this notebook kernel for the shimming checks:")
print(" ", kernel_display_name)

Install passive-shimming packages
---------------------------------
Packages:
  numpy>=2.0
  passive-shimming==0.0.1
  pymoo
  magpylib
  trimesh
  numpy-stl
  plotly
  matplotlib
  scipy
  jupyter
  ipykernel

Run these commands in Terminal / Command Prompt.
-----------------------------------------------

cd "/Users/sairamgeethanath/Documents/Contributions/Tools/Projects/adelpha/console/notebooks"

.diy-mri-shimming/bin/python -m pip install --upgrade pip setuptools wheel

.diy-mri-shimming/bin/python -m pip install \
  "numpy>=2.0" \
  "passive-shimming==0.0.1" \
  "pymoo" \
  "magpylib" \
  "trimesh" \
  "numpy-stl" \
  "plotly" \
  "matplotlib" \
  "scipy" \
  "jupyter" \
  "ipykernel"

.diy-mri-shimming/bin/python -m ipykernel install --user \
  --name diy-mri-shimming \
  --display-name "DIY MRI Passive Shimming Python 3.11"

.diy-mri-shimming/bin/python -c "import sys; print('Python executable:', sys.executable); print('Python version:', sys.version)"
.diy-mri-shimming/bin/pyth

## Verify `.diy-mri-shimming`

Select the kernel:

```text
DIY MRI Passive Shimming Python 3.11
```

Then restart the notebook kernel and run the next verification cells.

In [1]:
# Verify passive-shimming notebook kernel

import sys
import platform

print("Passive-shimming kernel verification")
print("------------------------------------")

print("Python executable:", sys.executable)
print("Python version   :", sys.version)
print("Platform         :", platform.platform())
print("mac_ver          :", platform.mac_ver())

expected_env_name = ".diy-mri-shimming"

if expected_env_name in sys.executable:
    print("\nPASS: This notebook is using the passive-shimming environment.")
else:
    print("\nWARNING: This notebook is not using the expected environment.")
    print("Select the kernel:")
    print("  DIY MRI Passive Shimming Python 3.11")
    print("Then restart the notebook kernel and rerun this cell.")

Passive-shimming kernel verification
------------------------------------
Python executable: /Users/sairamgeethanath/Documents/Contributions/Tools/Projects/adelpha/.diy-mri-shimming/bin/python
Python version   : 3.11.9 (v3.11.9:de54cf5be3, Apr  2 2024, 07:12:50) [Clang 13.0.0 (clang-1300.0.29.30)]
Platform         : macOS-26.6.2-arm64-arm-64bit
mac_ver          : ('26.6.2', ('', '', ''), 'arm64')

PASS: This notebook is using the passive-shimming environment.


In [2]:
# Passive-shimming package and workflow-module smoke test

print("Passive-shimming package smoke test")
print("-----------------------------------")

import sys
import importlib
import inspect
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

import numpy as np
import matplotlib
import scipy
import pymoo
import magpylib
import trimesh
import stl
import plotly

print("Python executable:", sys.executable)

print("\nCore packages")
print("-------------")
print("NumPy      :", np.__version__)
print("Matplotlib :", matplotlib.__version__)
print("SciPy      :", scipy.__version__)
print("pymoo      :", pymoo.__version__)
print("magpylib   :", magpylib.__version__)
print("trimesh    :", trimesh.__version__)
print("numpy-stl  : OK")
print("plotly     :", plotly.__version__)

major_numpy = int(np.__version__.split('.')[0])
if major_numpy < 2:
    print("\nWARNING: NumPy is <2. This environment is expected to use numpy>=2.0.")
else:
    print("\nPASS: Passive-shimming environment has numpy>=2.0.")

try:
    print("passive-shimming package version:", version("passive-shimming"))
except PackageNotFoundError:
    print("WARNING: passive-shimming package metadata not found.")

modules_to_check = [
    "shimming_io",
    "shimming_geometry",
    "shimming_basis",
    "shimming_optimization",
    "shimming_reporting",
    "shimming_export",
]

expected_objects = {
    "shimming_io": ["ShimmingRun", "FieldMap"],
    "shimming_geometry": ["TrayGeometry", "ShimMagnet", "CandidateTray"],
    "shimming_basis": ["ShimBasis"],
    "shimming_optimization": ["ShimOptimizationProblem", "ShimOptimizer"],
    "shimming_reporting": ["ShimmingReporter"],
    "shimming_export": ["ShimCollectionExporter", "ShimTrayExporter"],
}

all_ok = True

print("\nWorkflow module checks")
print("----------------------")

for module_name in modules_to_check:
    try:
        module = importlib.import_module(module_name)
        print(f"PASS: {module_name} imported from {getattr(module, '__file__', 'unknown')}")

        for object_name in expected_objects[module_name]:
            if hasattr(module, object_name):
                obj = getattr(module, object_name)
                print(f"      PASS: {object_name}")
                try:
                    print("            Signature:", inspect.signature(obj))
                except Exception:
                    pass
            else:
                all_ok = False
                print(f"      WARNING: {object_name} not found")

    except Exception as exc:
        all_ok = False
        print(f"FAIL: {module_name} import failed: {repr(exc)}")

external_stl = Path("./data/delta1_first_layer_shim_tray.STL")
print("\nExternal STL check")
print("------------------")
print("Expected path:", external_stl)
if external_stl.exists():
    print("PASS: External STL file exists.")
else:
    print("NOTE: External STL file not found at this path yet.")
    print("      This is okay for environment setup, but the shimming notebook will need it when USE_EXTERNAL_STL = True.")

if all_ok:
    print("\nPASS: Passive-shimming workflow modules are available.")
else:
    print("\nWARNING: One or more passive-shimming checks failed.")

Passive-shimming package smoke test
-----------------------------------
Python executable: /Users/sairamgeethanath/Documents/Contributions/Tools/Projects/adelpha/.diy-mri-shimming/bin/python

Core packages
-------------
NumPy      : 2.4.6
Matplotlib : 3.11.2
SciPy      : 1.17.1
pymoo      : 0.6.2
magpylib   : 5.2.3
trimesh    : 5.1.0
numpy-stl  : OK
plotly     : 7.0.0

PASS: Passive-shimming environment has numpy>=2.0.
passive-shimming package version: 0.0.1

Workflow module checks
----------------------
PASS: shimming_io imported from /Users/sairamgeethanath/Documents/Contributions/Tools/Projects/adelpha/.diy-mri-shimming/lib/python3.11/site-packages/shimming_io.py
      PASS: ShimmingRun
            Signature: (output_root, random_seed=42, stop_after_step=None, run_id=None)
      PASS: FieldMap
            Signature: (positions_m, B_T, gammabar_Hz_per_T, V=None, source_filename=None, grid_spacing_input=None, position_scale_to_m=1.0, field_scale_to_T=1.0)
PASS: shimming_geometry impor

## Setup complete

You are ready when:

### Acquisition / console environment

- Kernel selected: `DIY MRI Workshop Python 3.11`
- Python path includes `.diy-mri-workshop`
- NumPy is `1.x`
- PyPulseq imports successfully
- MRZeroCore imports successfully
- `pypulseq_smoke_test.seq` is written

### Passive-shimming environment

- Kernel selected: `DIY MRI Passive Shimming Python 3.11`
- Python path includes `.diy-mri-shimming`
- NumPy is `2.x`
- `passive-shimming==0.0.1` is installed
- `pymoo`, `magpylib`, `trimesh`, `numpy-stl`, and `plotly` import successfully
- `shimming_io`, `shimming_geometry`, `shimming_basis`, `shimming_optimization`, `shimming_reporting`, and `shimming_export` import successfully

You can now run the RF spin echo, RF pulse-width calibration, 1D SE, 2D SE + MRZero simulation, and passive-shimming notebooks.